In [2]:
import json
import subprocess
import sys
import tempfile
import os
import re
from tqdm.auto import tqdm

# --- CONFIGURATION ---
MODEL_SOLUTIONS_FILE = "/content/model_solutions_batched (2).jsonl"
GROUND_TRUTH_FILE = "/content/test_suite_200_FINAL.jsonl"

TOKEN_LIMITS = {"EASY": 2048, "MEDIUM": 4096, "HARD": 8192}

import json
import re

def parse_strict_response(full_text):
    # 1. ISOLATE THE GENERATION
    # We need to find where the prompt ends.
    # Usually, it's after "Now solve the problem and return the code."
    split_marker = "Now solve the problem and return the code."
    if split_marker in full_text:
        actual_gen = full_text.split(split_marker)[-1].strip()
    else:
        actual_gen = full_text # Fallback if marker is missing

    # 2. CHECK REAL TAGS (Only in the generated part)
    # We check for the presence of the tags in the model's actual answer
    has_think = "<think>" in actual_gen
    has_code = "<code>" in actual_gen

    # 3. EXTRACT CONTENT
    think_text = ""
    code_text = ""

    # Find what is between <think> and </think> (or <code>)
    think_match = re.search(r'<think>(.*?)(?:</think>|<code>|$)', actual_gen, re.DOTALL)
    if think_match:
        think_text = think_match.group(1).strip()

    code_match = re.search(r'<code>(.*?)(?:</code>|$)', actual_gen, re.DOTALL)
    if code_match:
        code_text = code_match.group(1).strip()

    total_len = len(actual_gen)
    density = (len(think_text) / total_len) if total_len > 0 else 0

    return {
        "far": 1.0 if (has_think and has_code) else 0.0,
        "density": density,
        "code": code_text,
        "has_code": has_code
    }

# Update the Judge loop to use 'parse_strict_response'

def run_python_sandbox(code, input_data, timeout=3):
    """Executes code in a subprocess with stdin."""
    with tempfile.NamedTemporaryFile(suffix='.py', delete=False, mode='w') as tmp:
        tmp.write(code)
        tmp_path = tmp.name
    try:
        result = subprocess.run(
            [sys.executable, tmp_path],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return result.stdout.strip(), result.stderr
    except subprocess.TimeoutExpired:
        return None, "TIMEOUT"
    except Exception as e:
        return None, str(e)
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

def evaluate_suite():
    # Load Truth
    truth = {json.loads(l)['id']: json.loads(l) for l in open(GROUND_TRUTH_FILE)}

    stats = {
        "EASY": {"pass": 0, "total": 0, "far": 0, "density": [], "tokens": []},
        "MEDIUM": {"pass": 0, "total": 0, "far": 0, "density": [], "tokens": []},
        "HARD": {"pass": 0, "total": 0, "far": 0, "density": [], "tokens": []}
    }

    with open(MODEL_SOLUTIONS_FILE, 'r') as f:
        for line in f:
            sol = json.loads(line)
            task_id = sol['id']
            if task_id not in truth: continue

            ref = truth[task_id]
            diff = ref['difficulty'].upper()
            parsed = parse_model_response(sol['model_output'])

            # --- METRIC 1: Format Adherence (FAR) ---
            # Correct if it has both start tags
            is_formatted = parsed['has_think'] and parsed['has_code']
            if is_formatted: stats[diff]["far"] += 1

            # --- METRIC 2: Reasoning Density ---
            # % of the output used for thinking vs total
            if parsed['total_chars'] > 0:
                density = len(parsed['think_text']) / parsed['total_chars']
                stats[diff]["density"].append(density)

            # --- METRIC 3: Pass@1 (Functional Correctness) ---
            passed_tests = False
            fail_reason = "NONE"

            # Check Token Limit
            if sol['tokens_used'] >= TOKEN_LIMITS.get(diff, 4096):
                fail_reason = "TOKEN_EXCEEDED"
            elif not parsed['has_code'] or not parsed['code_text']:
                fail_reason = "NO_CODE_FOUND"
            else:
                # Run Sandbox
                all_cases_passed = True
                for test in ref['test_cases']:
                    actual, err = run_python_sandbox(parsed['code_text'], test['input'])
                    if actual != test['output'].strip():
                        all_cases_passed = False
                        break
                passed_tests = all_cases_passed

            if passed_tests:
                stats[diff]["pass"] += 1

            stats[diff]["total"] += 1

    # --- FINAL DISPLAY ---
    print("\n" + "="*50)
    print(f"{'DIFFICULTY':<10} | {'PASS@1':<10} | {'FAR':<10} | {'THINK DENSITY'}")
    print("-" * 50)
    for d in ["EASY", "MEDIUM", "HARD"]:
        s = stats[d]
        if s['total'] == 0: continue

        pass_rate = (s['pass'] / s['total']) * 100
        far_rate = (s['far'] / s['total']) * 100
        avg_density = (sum(s['density']) / len(s['density'])) * 100 if s['density'] else 0

        print(f"{d:<10} | {pass_rate:>6.1f}%    | {far_rate:>6.1f}%    | {avg_density:>6.1f}%")
    print("="*50)

evaluate_suite()


DIFFICULTY | PASS@1     | FAR        | THINK DENSITY
--------------------------------------------------
EASY       |    0.0%    |  100.0%    |    0.9%
MEDIUM     |    0.0%    |  100.0%    |    0.9%
HARD       |    0.0%    |  100.0%    |    0.4%


In [3]:
import json
import re

def check_attempts(filename='/content/model_solutions_batched (2).jsonl'):
    stats = {"EASY": [0, 0, 0], "MEDIUM": [0, 0, 0], "HARD": [0, 0, 0]} # [Total, Thought, Coded]
    prompt_marker = "Now solve the problem and return the code."

    with open(filename, 'r') as f:
        for line in f:
            data = json.loads(line)
            diff = data['difficulty'].upper()
            output = data['model_output']

            # 1. Isolate the Model's actual response
            if prompt_marker in output:
                gen_part = output.split(prompt_marker)[-1].strip()
            else:
                gen_part = output.strip()

            stats[diff][0] += 1

            # 2. Check for Reasoning Attempt (At least 50 chars of text)
            if len(gen_part) > 50:
                stats[diff][1] += 1

            # 3. Check for Code Attempt
            # (Looking for 'def', 'import', 'print', or your <code> tag)
            code_keywords = ['def ', 'import ', 'print(', 'sys.', ' = ', '<code>']
            if any(key in gen_part for key in code_keywords):
                stats[diff][2] += 1

    print("=== BASELINE ATTEMPT REPORT ===")
    print(f"{'DIFF':<8} | {'TOTAL':<6} | {'THOUGHT?':<10} | {'CODE?':<8}")
    print("-" * 40)
    for d in ["EASY", "MEDIUM", "HARD"]:
        t, thought, coded = stats[d]
        t_pct = (thought/t*100) if t > 0 else 0
        c_pct = (coded/t*100) if t > 0 else 0
        print(f"{d:<8} | {t:<6} | {t_pct:>6.1f}%   | {c_pct:>6.1f}%")

check_attempts()

=== BASELINE ATTEMPT REPORT ===
DIFF     | TOTAL  | THOUGHT?   | CODE?   
----------------------------------------
EASY     | 60     |   83.3%   |   76.7%
MEDIUM   | 80     |   95.0%   |   88.8%
HARD     | 60     |   95.0%   |   83.3%


In [ ]:
# Structured Metric Analysis & Reasoning
# The baseline results reveal a model that is linguistically capable but structurally undisciplined. Here is the breakdown of the metrics and the technical reasoning for the values you observed.

# 1. Pass@1 (Functional Correctness) — 0.0%
# The Metric: Measures if the code generated by the model passes all hidden test cases on the first attempt.
# Reasoning for 0.0%: * The Parsing Gap: Your judge is programmed to extract code specifically from <code> tags. In the baseline state, the model writes code in a "Chat" style (plain text or Markdown).

# The Result: Because the code is not contained within the specific XML-style <code> block the judge expects, the judge finds "zero" code to execute. Effectively, a formatting failure is masking any underlying coding competence.

# 2. FAR (Format Adherence Rate) — 100.0% (False Positive)
# The Metric: Measures if the model correctly uses the <tags>, <think>, and <code> structure.
# Reasoning for 100%: * Pattern Echoing: In your inference script, the System Prompt explicitly contains the strings <think> and <code>.

# The Error: The initial judge detected these strings at the very top of the output (where the prompt is echoed) and marked it as "Adherent." In reality, the model's actual response (the part it generated) did not use these tags to wrap its logic. It is a "False Positive" caused by the model repeating the instructions back to the system.

# 3. Think Density — 0.4% to 0.9%
# The Metric: The ratio of characters inside the <think> block compared to the total output length.
# Reasoning for low values: * Tag Neglect: Since the model is not actually putting its reasoning inside the <think> tags, the "Think Content" is essentially empty strings in the eyes of the parser.

# Yapping in Plain Text: The model is generating thousands of tokens of reasoning, but it is doing so in regular prose. Because that prose is outside the tags, the density calculation (Inner Content / Total Length) results in a near-zero value.

# 4. Attempt Report — Thought? (83-95%) | Code? (76-88%)
# The Metric: A "loose" check to see if the model generated any reasoning text or any Python-like keywords (def, import).
# Reasoning for high values:

# Latent Knowledge: The 1.5B model was pre-trained on massive amounts of web data and code. It "knows" what a programming problem looks like and it "knows" how to write Python.

# The Disconnect: This proves the model has competence (it knows the subject) but lacks alignment (it doesn't follow your specific rules). It is trying to be a "Helpful Assistant" rather than a "Structured Reasoning Engine."

# Final Summary for your Professor
# The baseline model is currently "Knowledge-Rich but Structure-Poor." It attempts 95% of the problems and writes Python code in nearly 85% of them, but because it fails to wrap that code in the required <code> tags, its functional score (Pass@1) remains zero.

# Conclusion: The model's failure is not due to a lack of intelligence, but a failure to follow the interface constraints. This confirms that Stage 1 (SFT) is mandatory to force the model to adopt the <think><code> protocol.